In [1]:
import asyncio
import aiohttp
import time
import os
import json
import hashlib
import logging
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from scrapy_wrapper import ScrapySubprocessWrapper

# Constants from original pipeline
CATEGORY_THRESHOLDS = {
    "ABOUT_US": 200,
    "EBOOK": 200,
    "COURSES": 300,
    "RECENT_BLOG": 450,
    "TESTIMONIALS": 100,
    "WEBINAR": 150,
    "SERVICES": 150,
    "PODCAST": 200,
    "SHOP": 100,
}

CATEGORY_KEYWORDS = {
    "ABOUT_US": [
        "about", "who-we-are", "company", "our-story", "mission", "values", 
        "about-us", "story", "timeline", "milestones", "why-us"
    ],
    "EBOOK": [
        "ebook", "e-book", "whitepaper", "white-paper", "guide", "pdf", 
        "resources", "downloads", "books", "library", "documents"
    ],
    "COURSES": [
        "course", "academy", "learning", "training", "workshop",
        "certification", "program", "bootcamp", "masterclass", 
        "education", "class", "e-learning"
    ],
    "RECENT_BLOG": [
        "blog", "insights", "articles", "news", "updates", 
        "post", "media", "latest", "trends", "press", 
        "content-hub"
    ],
    "TESTIMONIALS": [
        "testimonial", "reviews", "case-study", "success-story", 
        "client-story", "customer-story", "feedback", "clients", 
        "portfolio", "results", "social-proof"
    ],
    "WEBINAR": [
        "webinar", "event", "session", "live", "virtual-event", 
        "presentation", "conference", "summit", "registration", 
        "upcoming", "schedule"
    ],
    "SERVICES": [
        "service", "solution", "offering", "expertise", "consulting", 
        "what-we-do", "capability", "support", "practice", 
        "professional-services", "how-we-help"
    ],
    "PODCAST": [
        "podcast", "episodes", "audio", "listen", "show", 
        "interview", "series", "stream", "speakers", 
        "voice", "subscribe"
    ],
    "SHOP": [
        "shop", "store", "buy", "purchase", "products", 
        "cart", "checkout", "pricing", "e-commerce", 
        "merchandise", "order"
    ]
}

COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M",
    "EBOOK": "N",
    "COURSES": "O",
    "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q",
    "WEBINAR": "R",
    "SERVICES": "S",
    "PODCAST": "T",
    "SHOP": "U"
}

CATEGORY_RULES = {
    'ABOUT_US': 'ascending',
    'EBOOK': 'ascending',
    'COURSES': 'ascending',
    'RECENT_BLOG': 'descending',
    'TESTIMONIALS': 'ascending',
    'WEBINAR': 'descending',
    'SERVICES': 'descending',
    'PODCAST': 'descending',
    'SHOP': 'ascending'
}

EXTRACTION_METADATA_COLUMN = "V"
CACHE_DIR = "scrapy_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

@dataclass
class ProcessingResult:
    category: str
    content: str
    metadata: str
    urls_found: List[str]

@dataclass
class ProductionConfig:
    max_concurrent_rows: int = 3
    max_concurrent_scrapes_per_row: int = 2
    max_connections_per_host: int = 10
    google_sheets_batch_size: int = 10
    google_sheets_rate_limit: float = 1.0
    scraping_timeout_seconds: int = 20
    worker_timeout_seconds: int = 300
    max_retries: int = 2
    retry_delay_seconds: float = 1.0
    max_content_size_bytes: int = 50000
    max_cache_size: int = 1000

class ProductionMonitor:
    def __init__(self):
        self.stats = defaultdict(int)
        self.start_time = time.time()
        self.errors = []
        logging.basicConfig(
            level=logging.DEBUG,  # Increased to DEBUG
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('pipeline.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def log_progress(self, processed: int, total: int):
        elapsed = time.time() - self.start_time
        rate = processed / elapsed if elapsed > 0 else 0
        eta = (total - processed) / rate if rate > 0 else 0
        self.logger.info(f"Progress: {processed}/{total} ({processed/total*100:.1f}%) "
                        f"Rate: {rate:.2f} rows/sec, ETA: {eta/60:.1f} min")

    def log_error(self, error: str, row_num: Optional[int] = None):
        error_msg = f"Row {row_num}: {error}" if row_num else error
        self.errors.append(error_msg)
        self.logger.error(error_msg)

    def log_stats(self):
        self.logger.info("Pipeline Statistics:")
        for key, value in self.stats.items():
            self.logger.info(f"  {key}: {value}")
        if self.errors:
            self.logger.error(f"Errors encountered: {len(self.errors)}")
            for error in self.errors[-10:]:
                self.logger.error(f"  {error}")

def calculate_url_depth(url: str) -> int:
    try:
        parsed = urlparse(url)
        path = parsed.path.strip('/').split('/')
        return len(path)
    except Exception:
        return -1

def truncate_to_bytes(text: str, max_bytes: int) -> str:
    encoded = text.encode('utf-8')
    if len(encoded) <= max_bytes:
        return text
    truncated = encoded[:max_bytes]
    return truncated.decode('utf-8', errors='ignore')

def safe_join(contents: List[str], max_bytes: int = 50000, delimiter: str = " --- NEXT CONTENT FROM HERE --- ") -> str:
    final_text = ""
    for content in contents:
        candidate = final_text + (delimiter if final_text else "") + content
        if len(candidate.encode('utf-8')) > max_bytes:
            break
        final_text = candidate
    return final_text

def extract_main_html_content(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article")
    if main:
        return main.get_text(separator="\n", strip=True)
    candidates = [
        div for div in soup.find_all("div")
        if len(div.get_text(strip=True)) > 200
           and not any(c in " ".join(div.get("class", [])).lower() for c in ["nav", "header", "footer", "popup"])
    ]
    if candidates:
        return max(candidates, key=lambda d: len(d.get_text(strip=True))).get_text(separator="\n", strip=True)
    return soup.get_text(separator="\n", strip=True)

class GoogleSheetsManager:
    def __init__(self, credentials_file: str):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        import re
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError("Invalid Google Sheet URL")

    def get_urls(self, spreadsheet_id: str, start_row: int = 2) -> List[Tuple[int, str]]:
        range_name = f"G{start_row}:G"
        result = self.service.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()
        values = result.get('values', [])
        return [(i + start_row, row[0]) for i, row in enumerate(values) if row and row[0].strip()]

    def batch_update_cells(self, spreadsheet_id: str, updates: List[Dict]):
        if not updates:
            return
        body = {
            'valueInputOption': 'RAW',
            'data': updates
        }
        self.service.spreadsheets().values().batchUpdate(
            spreadsheetId=spreadsheet_id, 
            body=body
        ).execute()

class OptimizedPipeline:
    def __init__(self, config: ProductionConfig, credentials_file: str):
        self.config = config
        self.sheet_mgr = GoogleSheetsManager(credentials_file)
        self.monitor = ProductionMonitor()
        self.scrapy_wrapper = ScrapySubprocessWrapper('site_spider')
        self.connector = aiohttp.TCPConnector(
            limit=100,
            limit_per_host=config.max_connections_per_host,
            keepalive_timeout=30,
            enable_cleanup_closed=True
        )
        self.session = None
        self.task_queue = asyncio.Queue()
        self.results_queue = asyncio.Queue()

    async def __aenter__(self):
        self.session = aiohttp.ClientSession(connector=self.connector)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session:
            await self.session.close()
        await self.connector.close()

    def filter_by_category(self, urls: List[str], category: str) -> List[str]:
        keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
        if not keywords:
            self.monitor.logger.debug(f"No keywords defined for category {category}")
            return []
        filtered = [u for u in urls if any(k in u.lower() for k in keywords)]
        self.monitor.logger.debug(f"Filtered {len(filtered)} URLs for category {category}")
        return filtered

    def _hash_url(self, url: str) -> str:
        return hashlib.md5(url.encode()).hexdigest()

    def _get_cache_path(self, url: str) -> str:
        return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")

    def get_cached_result(self, url: str) -> Optional[List[str]]:
        cache_path = self._get_cache_path(url)
        if os.path.exists(cache_path):
            try:
                with open(cache_path, 'r') as f:
                    links = json.load(f)
                    self.monitor.logger.debug(f"Cache hit for {url}: {len(links)} URLs")
                    return links
            except Exception as e:
                self.monitor.logger.error(f"Cache read error for {url}: {str(e)}")
                return None
        self.monitor.logger.debug(f"No cache found for {url}")
        return None

    async def scrape_url_with_session(self, url: str) -> str:
        self.monitor.logger.debug(f"Scraping URL: {url}")
        try:
            async with self.session.get(url, timeout=aiohttp.ClientTimeout(total=self.config.scraping_timeout_seconds)) as response:
                if response.status == 200:
                    html = await response.text()
                    content = extract_main_html_content(html)
                    self.monitor.logger.debug(f"Successfully scraped {url}: {len(content)} chars")
                    return content
                else:
                    self.monitor.logger.error(f"HTTP error {response.status} for {url}")
                    return f"Error: HTTP {response.status}"
        except Exception as e:
            self.monitor.logger.error(f"Scraping error for {url}: {str(e)}")
            return f"Error: {str(e)}"

    async def scrape_multiple_urls(self, urls: List[str]) -> List[str]:
        if not urls:
            self.monitor.logger.debug("No URLs to scrape")
            return []
        semaphore = asyncio.Semaphore(self.config.max_concurrent_scrapes_per_row)
        async def scrape_with_retry(url: str) -> str:
            async with semaphore:
                for attempt in range(self.config.max_retries):
                    try:
                        result = await self.scrape_url_with_session(url)
                        if not result.startswith("Error:"):
                            self.monitor.stats['scraping_success'] += 1
                            return result
                    except Exception as e:
                        if attempt == self.config.max_retries - 1:
                            self.monitor.stats['scraping_failures'] += 1
                            self.monitor.logger.error(f"Max retries reached for {url}: {str(e)}")
                            return f"Error: {str(e)}"
                        await asyncio.sleep(self.config.retry_delay_seconds)
                return f"Error: Max retries exceeded"
        tasks = [scrape_with_retry(url) for url in urls]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        return [str(r) if isinstance(r, Exception) else r for r in results]

    async def process_single_category(self, sub_urls: List[str], category: str) -> ProcessingResult:
        self.monitor.logger.debug(f"Processing category {category} with {len(sub_urls)} URLs")
        filtered_urls = self.filter_by_category(sub_urls, category)
        if not filtered_urls:
            self.monitor.logger.debug(f"No URLs matched for category {category}")
            return ProcessingResult(
                category=category,
                content="No URL found",
                metadata=f"{category.upper()}=0",
                urls_found=[]
            )
        url_depth_pairs = [(url, calculate_url_depth(url)) for url in filtered_urls]
        url_depth_pairs = [(url, depth) for url, depth in url_depth_pairs if depth != -1]
        if not url_depth_pairs:
            self.monitor.logger.debug(f"No valid URL depths for category {category}")
            return ProcessingResult(
                category=category,
                content="No URL found",
                metadata=f"{category.upper()}=0",
                urls_found=[]
            )
        sort_order = CATEGORY_RULES.get(category.upper(), 'ascending')
        reverse_sort = sort_order == 'descending'
        sorted_pairs = sorted(url_depth_pairs, key=lambda x: x[1], reverse=reverse_sort)
        selected_urls = [url for url, _ in sorted_pairs[:10]]
        self.monitor.logger.debug(f"Selected {len(selected_urls)} URLs for category {category}")
        contents = await self.scrape_multiple_urls(selected_urls)
        valid_contents = [c for c in contents if c and not c.startswith("Error:")]
        if not valid_contents:
            content = "No meaningful content found"
            metadata = f"{category.upper()}=0"
            self.monitor.logger.debug(f"No valid content for category {category}")
        else:
            content = safe_join(valid_contents, self.config.max_content_size_bytes)
            metadata = f"{category.upper()}={len(valid_contents)}"
            self.monitor.logger.debug(f"Extracted {len(valid_contents)} valid contents for category {category}")
        return ProcessingResult(
            category=category,
            content=content,
            metadata=metadata,
            urls_found=selected_urls
        )

    async def process_all_categories_for_row(self, row_num: int, main_url: str) -> Dict[str, ProcessingResult]:
        self.monitor.logger.info(f"Processing all categories for row {row_num}: {main_url}")
        sub_urls = self.get_cached_result(main_url)
        if not sub_urls:
            self.monitor.logger.info(f"No cache hit for {main_url}, running Scrapy")
            sub_urls = self.scrapy_wrapper.process_urls_batch([main_url])
            self.monitor.logger.debug(f"Scrapy returned {len(sub_urls)} URLs for {main_url}")
            if sub_urls:
                cache_path = self._get_cache_path(main_url)
                self.monitor.logger.debug(f"Caching {len(sub_urls)} URLs to {cache_path}")
                with open(cache_path, 'w') as f:
                    json.dump(sub_urls, f, indent=2)
        
        if not sub_urls:
            self.monitor.logger.error(f"No URLs retrieved for {main_url}")
            empty_results = {}
            for category in CATEGORY_KEYWORDS.keys():
                empty_results[category] = ProcessingResult(
                    category=category,
                    content="No URL found",
                    metadata=f"{category}=0",
                    urls_found=[]
                )
            return empty_results
        
        tasks = []
        for category in CATEGORY_KEYWORDS.keys():
            task = self.process_single_category(sub_urls, category)
            tasks.append((category, task))
        
        results = {}
        for category, task in tasks:
            try:
                result = await task
                results[category] = result
            except Exception as e:
                self.monitor.log_error(f"Error processing category {category} for row {row_num}: {e}", row_num)
                results[category] = ProcessingResult(
                    category=category,
                    content=f"Error: {str(e)}",
                    metadata=f"{category}=0",
                    urls_found=[]
                )
        return results

    async def worker(self, worker_id: int):
        processed_count = 0
        while True:
            try:
                row_data = await asyncio.wait_for(
                    self.task_queue.get(),
                    timeout=self.config.worker_timeout_seconds
                )
                if row_data is None:
                    break
                row_num, main_url = row_data
                start_time = time.time()
                try:
                    results = await self.process_all_categories_for_row(row_num, main_url)
                    processing_time = time.time() - start_time
                    await self.results_queue.put((row_num, results))
                    processed_count += 1
                    self.monitor.stats['rows_processed'] += 1
                    self.monitor.stats['total_processing_time'] += processing_time
                    if processed_count % 5 == 0:
                        self.monitor.logger.info(f"Worker {worker_id} processed {processed_count} rows")
                except Exception as e:
                    self.monitor.log_error(f"Worker {worker_id} failed on row {row_num}: {e}", row_num)
                    self.monitor.stats['worker_failures'] += 1
                finally:
                    self.task_queue.task_done()
            except asyncio.TimeoutError:
                self.monitor.log_error(f"Worker {worker_id} timeout")
                break
            except Exception as e:
                self.monitor.log_error(f"Worker {worker_id} unexpected error: {e}")
                break

    async def batch_update_sheets(self, batch_results: List[Tuple[int, Dict[str, ProcessingResult]]]):
        if not batch_results:
            self.monitor.logger.debug("No batch results to update")
            return
        updates = []
        for row_num, results in batch_results:
            for category, result in results.items():
                column = COLUMN_TO_WRITE_URL_TO.get(category)
                if column:
                    updates.append({
                        'range': f"{column}{row_num}",
                        'values': [[result.content]]
                    })
            all_metadata = [result.metadata for result in results.values()]
            combined_metadata = ','.join(all_metadata)
            updates.append({
                'range': f"{EXTRACTION_METADATA_COLUMN}{row_num}",
                'values': [[combined_metadata]]
            })
        try:
            self.sheet_mgr.batch_update_cells(self.spreadsheet_id, updates)
            self.monitor.stats['sheets_updates'] += len(batch_results)
            self.monitor.logger.debug(f"Updated {len(batch_results)} rows in Google Sheets")
        except Exception as e:
            self.monitor.log_error(f"Batch sheets update failed: {e}")

    async def process_all_optimized(self, sheet_url: str, start_row: int = 2):
        try:
            self.monitor.logger.info("Starting optimized pipeline")
            self.spreadsheet_id = self.sheet_mgr.extract_spreadsheet_id(sheet_url)
            urls = self.sheet_mgr.get_urls(self.spreadsheet_id, start_row)
            self.monitor.logger.info(f"Processing {len(urls)} rows with {self.config.max_concurrent_rows} workers")
            for row_num, main_url in urls:
                self.monitor.logger.debug(f"Queueing row {row_num}: {main_url}")
                await self.task_queue.put((row_num, main_url))
            workers = []
            for i in range(self.config.max_concurrent_rows):
                worker_task = asyncio.create_task(self.worker(i))
                workers.append(worker_task)
            results_processor = asyncio.create_task(self.process_results(len(urls)))
            await self.task_queue.join()
            for _ in workers:
                await self.task_queue.put(None)
            await asyncio.gather(*workers, return_exceptions=True)
            await results_processor
            self.monitor.log_stats()
            self.monitor.logger.info("Pipeline completed successfully")
        except Exception as e:
            self.monitor.log_error(f"Pipeline failed: {e}")
            raise

    async def process_results(self, total_rows: int):
        processed = 0
        batch_results = []
        while processed < total_rows:
            try:
                row_num, results = await asyncio.wait_for(
                    self.results_queue.get(),
                    timeout=self.config.worker_timeout_seconds
                )
                batch_results.append((row_num, results))
                processed += 1
                if processed % 5 == 0 or processed == total_rows:
                    self.monitor.log_progress(processed, total_rows)
                if len(batch_results) >= self.config.google_sheets_batch_size or processed == total_rows:
                    self.monitor.logger.info(f"📝 WRITING BATCH: {len(batch_results)} rows to Google Sheets")
                    await self.batch_update_sheets(batch_results)
                    self.monitor.logger.info(f"✅ BATCH WRITTEN: Rows {[r[0] for r in batch_results]} updated in sheets")
                    batch_results = []
                    await asyncio.sleep(self.config.google_sheets_rate_limit)
            except asyncio.TimeoutError:
                self.monitor.log_error("Timeout waiting for results")
                break

In [2]:
CREDENTIALS_FILE = r"C:\Users\apwbm\OneDrive\Desktop\PROJECTS\P1\LEAD ENRICHMENT\lead-enricher-ai-be\data\url-to-email-445616-cebe4868914f.json"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/152clSgMA-9oW1xFotbToo0ci8DjgaWQDxYNL6Smpn9M/edit?gid=0#gid=0"

In [3]:
async def main():
    config = ProductionConfig(
        max_concurrent_rows=3,
        max_concurrent_scrapes_per_row=2,
        google_sheets_batch_size=10,
        google_sheets_rate_limit=1.0,
        scraping_timeout_seconds=20,
        worker_timeout_seconds=300,
        max_retries=2,
        retry_delay_seconds=1.0,
        max_content_size_bytes=50000,
        max_cache_size=1000
    )
    async with OptimizedPipeline(
        config=config,
        credentials_file=CREDENTIALS_FILE
    ) as pipeline:
        await pipeline.process_all_optimized(
            sheet_url=GOOGLE_SHEET_URL,
            start_row=2
        )



In [ ]:
# Run the async main function in the notebook
await main()

2025-09-24 20:38:01,745 - INFO - Starting optimized pipeline
2025-09-24 20:38:01,961 - DEBUG - URL being requested: GET https://sheets.googleapis.com/v4/spreadsheets/152clSgMA-9oW1xFotbToo0ci8DjgaWQDxYNL6Smpn9M/values/G2%3AG?alt=json
2025-09-24 20:38:01,970 - DEBUG - Making request: POST https://oauth2.googleapis.com/token
2025-09-24 20:38:03,955 - INFO - Processing 10 rows with 3 workers
2025-09-24 20:38:03,955 - DEBUG - Queueing row 2: https://www.marcusmillichap.com/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 3: https://www.axiscapital.com/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 4: https://www.aig.com/home
2025-09-24 20:38:03,955 - DEBUG - Queueing row 5: https://www.attuneinsurance.com/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 6: https://www.the-ros.com/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 7: https://www.nextgiv.com/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 8: https://stanfordhealthcare.org/
2025-09-24 20:38:03,955 - DEBUG - Queueing row 9: https: